# Colab self-supervised training and resume

Requires real ZuCo and verified alignment. No scientific results are bundled. Run CPU tests before training. See ../README.md and ../docs/.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess, sys
REPO_URL = 'https://github.com/ziadmohamdd/Neurotext-EEG-Representation-Learning.git'
REF = 'main'  # Set to your committed implementation revision; push these files first.
PROJECT = Path('/content/New_trends_AI_project1')
if not PROJECT.exists():
    subprocess.run(['git','clone',REPO_URL,str(PROJECT)],check=True)
subprocess.run(['git','checkout',REF],cwd=PROJECT,check=True)
os.chdir(PROJECT)
if not (PROJECT/'pyproject.toml').is_file():
    raise RuntimeError('Push the implemented package to the selected Git revision first.')
subprocess.run([sys.executable,'-m','pip','install','-r','requirements-colab.txt'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e','.'],check=True)
DRIVE = Path('/content/drive/MyDrive/eeg-project')
DRIVE.mkdir(parents=True,exist_ok=True)
from eeg_decoding.utils.device import device_info
print(device_info()[1])
# GPU availability varies. Heavy schedules belong on a runtime with sufficient free VRAM.

In [ ]:
from eeg_decoding.utils.storage import stage_manifest
from eeg_decoding.cli import main
SOURCE_MANIFEST = DRIVE/'data/manifests/combined_clean.jsonl'
TEXT_CACHE = DRIVE/'data/processed/sentence_embeddings.npz'
if not SOURCE_MANIFEST.is_file():
    raise FileNotFoundError('Acquire/align/preprocess ZuCo once in Drive before training.')
MANIFEST = stage_manifest(SOURCE_MANIFEST, '/content/eeg-staged')
CONFIG = 'configs/colab.yaml'
CHECKPOINTS = DRIVE/'checkpoints'
CHECKPOINTS.mkdir(exist_ok=True)
RESULTS = DRIVE/'results'
RESULTS.mkdir(exist_ok=True)

In [ ]:
OUTPUT = CHECKPOINTS/'ssl'
args = ['train-ssl','--manifest',str(MANIFEST),'--config',CONFIG,'--output',str(OUTPUT)]
if (OUTPUT/'last.pt').is_file():
    args += ['--resume',str(OUTPUT/'last.pt')]
main(args)

Keep the same preprocessing, split seed, config and epoch schedule when resuming. Checkpoints and history are written to Drive. Use a new output directory for a different experiment.